# Ply GPU worker on Kaggle (T4)

Processes captures sent from the Ply web app (ricco.codes/ply/capture.html). It polls the Ply API, claims the oldest queued
job, runs the pipeline, uploads `scene.spz` + floor data, and repeats. It stops after 45 minutes with no work, so GPU hours
are not burned idling.

**Inputs (Datasets, both private):**
- `ply-pipeline`: the repo's `pipeline/` folder.
- `ply-worker-config`: `config.json` with `{"api": "https://ply-api.<you>.workers.dev", "token": "<WORKER_TOKEN>"}`.
- optional `ply-wheels`: prebuilt CUDA wheels from an earlier run (saves ~15 min).

**Settings:** Accelerator GPU T4 x1 (not P100), Internet on. Run all, or push with `kaggle kernels push` (runs unattended).

In [ ]:
import glob, os, subprocess, sys, json, shutil, pathlib
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none")
assert gpu, "No GPU: enable Accelerator -> GPU T4 x1"
if "P100" in gpu:
    raise SystemExit("P100 (sm_60) is not supported by torch 2.11 cu128 wheels. Switch the accelerator to T4.")
ARCH = "7.5" if "T4" in gpu else ("8.9" if ("L4" in gpu or "4090" in gpu) else "7.5;8.0;8.6;8.9")
print("TORCH_CUDA_ARCH_LIST =", ARCH, "| python", sys.version.split()[0])

In [ ]:
# pipeline source -> /kaggle/working/pipeline (writable, editable install)
def find_input(rel, must=None):
    """Kaggle has moved dataset mounts before (/kaggle/input/<slug>/ vs nested owner folders); search instead of assuming."""
    hits = [p for p in glob.glob(f"/kaggle/input/**/{rel}", recursive=True) if must is None or must(p)]
    if not hits:
        for root, dirs, files in os.walk("/kaggle/input"):
            depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
            if depth <= 3:
                print("  ", root, sorted(files)[:8])
    return sorted(hits, key=len)[0] if hits else None
src = find_input("pyproject.toml", lambda p: os.path.isdir(os.path.join(os.path.dirname(p), "plypipe")))
assert src, "Add the Ply repo's pipeline/ folder as a Dataset named ply-pipeline"
PIPE = "/kaggle/working/pipeline"
if not os.path.exists(PIPE):
    shutil.copytree(os.path.dirname(src), PIPE, ignore=shutil.ignore_patterns(".venv", "work", "__pycache__"))
wheels = find_input("gsplat-*.whl")
env = dict(os.environ, TORCH_CUDA_ARCH_LIST=ARCH, PLY_GSPLAT_DIR="/kaggle/working/gsplat", MAX_JOBS="4", PYTHON=sys.executable)
if wheels:
    env["PLY_WHEELS"] = os.path.dirname(wheels); print("using prebuilt wheels from", env["PLY_WHEELS"])
else:
    env["PLY_SAVE_WHEELS"] = "/kaggle/working/wheels"; print("building CUDA wheels; they will be saved to /kaggle/working/wheels")
r = subprocess.run(["bash", f"{PIPE}/scripts/install_gpu.sh"], env=env)
assert r.returncode == 0, "install failed (scroll up)"

In [ ]:
# Node.js for splat-transform (SPZ export)
if shutil.which("npx") is None:
    subprocess.run("curl -fsSL https://nodejs.org/dist/v22.14.0/node-v22.14.0-linux-x64.tar.xz | tar -xJ -C /kaggle/working", shell=True, check=True)
    os.environ["PATH"] = "/kaggle/working/node-v22.14.0-linux-x64/bin:" + os.environ["PATH"]
print(subprocess.run(["node", "-v"], capture_output=True, text=True).stdout)

In [ ]:
# ---- run the worker ----
IDLE_EXIT_MIN = 45
cfg_path = find_input("config.json", lambda p: "token" in open(p).read())
assert cfg_path, "Add the private dataset ply-worker-config with config.json {api, token}"
cfg = json.load(open(cfg_path))
wenv = dict(os.environ, PLY_API=cfg["api"], PLY_WORKER_TOKEN=cfg["token"], PLY_WORKER_NAME="kaggle-" + gpu.split(",")[0].replace(" ", "-"),
            PLY_WORK_DIR="/kaggle/working/work", PLY_GSPLAT_DIR="/kaggle/working/gsplat", PLY_IDLE_EXIT_MIN=str(IDLE_EXIT_MIN),
            PYTHONPATH=PIPE + os.pathsep + os.environ.get("PYTHONPATH", ""))
p = subprocess.Popen([sys.executable, "-m", "plypipe.worker"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=PIPE, env=wenv)
for line in p.stdout:
    try:
        e = json.loads(line)
        if e.get("event") in ("start", "done", "skip", "warn", "error"):
            print(f"[{e['stage']}] {e['event']} {e.get('msg', '')}"[:300], flush=True)
    except ValueError:
        print(line.rstrip()[:300], flush=True)
print("worker exited with", p.wait())